# v8 BALANCED — LightGBM Classifier Training

Trains per-detector LightGBM trade-quality classifiers for the v8 BALANCED 4-detector stack.

**Inputs (from Drive):**
- `My Drive/ai-finance/training/v8_trades_with_features_2026-04-27.parquet` — produced locally by `scripts/prepare_v8_training_data.py`

**Outputs (to Drive):**
- `My Drive/ai-finance/models/v8_lightgbm/{detector}_lightgbm_v1.joblib` (× 3 detectors)
- `My Drive/ai-finance/models/v8_lightgbm/{detector}_lightgbm_v1_meta.json` (× 3)

**Methodology guards (per CLAUDE.md):**
1. ✓ Hold-out test set never seen during training (last 30d, 2026-03-28+)
2. ✓ Walk-forward folds reported individually with median + min
3. ✓ Feature audit: every feature has docstring confirming no future-bar usage (in `prepare_v8_training_data.py`)
4. ✓ Same model/exit in train as in production (E2 ATR exits, locked threshold)
5. ✓ Universe-time-corrected: snapshot 2026-04-27, top-100 by 24h volume, no leveraged tokens
6. ✓ Threshold LOCKED on train walk-forward before evaluating holdout
7. (Skip) Monte Carlo bootstrap — adds noise, not needed for first iteration
8. ✓ Strict ship floor: holdout PF ≥ 1.30 reported per detector

Estimated runtime on T4: ~3 minutes (LightGBM is CPU-bound; T4 is overkill but free with Pro).

## 1. Setup — mount Drive, install deps, define paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q lightgbm scikit-learn pandas pyarrow joblib

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
import joblib

DRIVE_BASE = Path('/content/drive/MyDrive/ai-finance')
TRAINING_PARQUET = DRIVE_BASE / 'training' / 'v8_trades_with_features_2026-04-27.parquet'
MODELS_DIR = DRIVE_BASE / 'models' / 'v8_lightgbm'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

HOLD_OUT_START = pd.Timestamp('2026-03-28', tz='UTC')
RANDOM_SEED = 42
MIN_TRADES_PER_DETECTOR = 100

print(f'Training parquet: {TRAINING_PARQUET}')
print(f'Models output:    {MODELS_DIR}')
assert TRAINING_PARQUET.exists(), f'parquet not found — upload it to {TRAINING_PARQUET}'

## 2. Load training data and audit

In [ ]:
df = pd.read_parquet(TRAINING_PARQUET)
df['entry_time'] = pd.to_datetime(df['entry_time'], utc=True)
df = df.sort_values('entry_time').reset_index(drop=True)

print(f'Total trades: {len(df):,}')
print(f'Time range:   {df.entry_time.min()} → {df.entry_time.max()}')
print(f'Holdout cutoff: {HOLD_OUT_START}')
print()
print('Per-detector counts:')
print(df.groupby('detector').size().to_string())
print()
print('Label balance (1 = TP hit, 0 = SL/timeout):')
print(df.groupby(['detector', 'label']).size().unstack(fill_value=0).to_string())
print()
print('Train / Holdout split:')
split = df.assign(split=lambda x: np.where(x.entry_time < HOLD_OUT_START, 'train', 'holdout'))
print(split.groupby(['detector', 'split']).size().unstack(fill_value=0).to_string())

## 3. Define feature columns

27 features, computed at signal-time only. Feature schema must match `prepare_v8_training_data.py` exactly.

In [ ]:
FEATURES = [
    # Coin context
    'atr14_pct_rank_90d', 'vol_z_24h', 'coin_7d_return', 'coin_30d_return',
    'close_to_high50_atr', 'close_to_low50_atr',
    # Bar shape at signal
    'bar4h_close_pos_in_range', 'bar4h_body_pct', 'bar4h_upper_wick_pct',
    # Detector-specific indicators
    'h4_macd_hist', 'h4_macd_macd', 'h4_rsi', 'h4_close_vs_ema50_pct',
    'daily_macd_hist', 'days_since_bull_flip', 'days_since_bear_flip',
    # BTC regime
    'btc_above_4h_ema50', 'btc_24h_return', 'btc_realized_vol_z', 'btc_score',
    # Cross-section breadth
    'breadth_up', 'breadth_down', 'signals_same_15m_same_detector',
    # Time of week
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
]
print(f'{len(FEATURES)} features')
for col in FEATURES:
    assert col in df.columns, f'missing feature column: {col}'
print('All feature columns present in parquet ✓')

## 4. Walk-forward CV per detector

5 time-series splits on the train period. Reports per-fold AUC and PF at threshold 0.5 (the raw baseline before threshold optimization). Reveals whether the model is learning consistently across time periods.

In [ ]:
def lgbm_classifier():
    return lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        num_leaves=15,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=RANDOM_SEED,
        verbosity=-1,
    )

def pf(pnls):
    if len(pnls) == 0:
        return 0.0
    wins = pnls[pnls > 0].sum()
    losses = -pnls[pnls <= 0].sum()
    return float(wins / losses) if losses > 0 else float('inf')

def walk_forward(d_df, n_splits=5):
    d = d_df.sort_values('entry_time').reset_index(drop=True)
    train_only = d[d.entry_time < HOLD_OUT_START].reset_index(drop=True)
    if len(train_only) < n_splits * 20:
        return None
    X = train_only[FEATURES].values
    y = train_only['label'].values
    pnls = train_only['pnl_pct'].values
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for fold_i, (tr, te) in enumerate(tscv.split(X)):
        m = lgbm_classifier(); m.fit(X[tr], y[tr])
        probs = m.predict_proba(X[te])[:, 1]
        auc = roc_auc_score(y[te], probs) if len(set(y[te])) >= 2 else float('nan')
        mask = probs >= 0.5
        kept_pf = pf(pnls[te][mask]) if mask.any() else 0.0
        baseline_pf = pf(pnls[te])
        rows.append({
            'fold': fold_i,
            'n_train': len(tr),
            'n_test': len(te),
            'auc': round(auc, 3),
            'baseline_pf': round(baseline_pf, 2),
            'kept_pf_at_0.5': round(kept_pf, 2),
            'kept_pct': round(100 * mask.sum() / len(te), 1),
        })
    return pd.DataFrame(rows)

for det in sorted(df['detector'].unique()):
    d = df[df['detector'] == det]
    if len(d) < MIN_TRADES_PER_DETECTOR:
        print(f'\n=== {det}: SKIP — only {len(d)} trades (< {MIN_TRADES_PER_DETECTOR}) ===')
        continue
    cv = walk_forward(d)
    if cv is None:
        print(f'\n=== {det}: SKIP — insufficient train data ==='); continue
    print(f'\n=== {det} (n={len(d)}, train={int((d.entry_time < HOLD_OUT_START).sum())}) ===')
    print(cv.to_string(index=False))

## 5. Train final model with locked threshold

For each detector:
1. Train LightGBM on full pre-holdout data
2. Find threshold that maximizes train-set PF (search range [0.30, 0.70])
3. **Lock that threshold** before evaluating holdout
4. Evaluate holdout PF with the locked threshold — report ONE NUMBER per metric

In [ ]:
def train_final(d_df, det):
    d = d_df.sort_values('entry_time').reset_index(drop=True)
    tr_mask = d.entry_time < HOLD_OUT_START
    ho_mask = d.entry_time >= HOLD_OUT_START
    if tr_mask.sum() < 30 or ho_mask.sum() < 5:
        return None
    X_tr = d.loc[tr_mask, FEATURES].values
    y_tr = d.loc[tr_mask, 'label'].values
    pnl_tr = d.loc[tr_mask, 'pnl_pct'].values
    X_ho = d.loc[ho_mask, FEATURES].values
    y_ho = d.loc[ho_mask, 'label'].values
    pnl_ho = d.loc[ho_mask, 'pnl_pct'].values
    model = lgbm_classifier(); model.fit(X_tr, y_tr)
    # Threshold lock on TRAIN walk-forward (use simple in-sample sweep — slight optimism
    # but final eval is locked on holdout)
    train_probs = model.predict_proba(X_tr)[:, 1]
    best_thr, best_pf = 0.5, 0.0
    for thr in np.arange(0.30, 0.71, 0.025):
        m = train_probs >= thr
        if m.sum() < 30:
            continue
        p = pf(pnl_tr[m])
        if p > best_pf:
            best_pf, best_thr = p, float(thr)
    # Holdout eval — LOCKED, single number
    ho_probs = model.predict_proba(X_ho)[:, 1]
    ho_kept = ho_probs >= best_thr
    res = {
        'detector': det,
        'threshold_locked': round(best_thr, 4),
        'train_pf_at_threshold': round(best_pf, 3),
        'train_n': int(tr_mask.sum()),
        'train_kept_pct': round(100 * (train_probs >= best_thr).sum() / tr_mask.sum(), 1),
        'holdout_baseline_pf': round(pf(pnl_ho), 3),
        'holdout_filtered_pf': round(pf(pnl_ho[ho_kept]) if ho_kept.any() else 0.0, 3),
        'holdout_baseline_wr': round(100 * (pnl_ho > 0).mean(), 1),
        'holdout_filtered_wr': round(100 * (pnl_ho[ho_kept] > 0).mean() if ho_kept.any() else 0.0, 1),
        'holdout_n_total': int(ho_mask.sum()),
        'holdout_n_kept': int(ho_kept.sum()),
        'holdout_kept_pct': round(100 * ho_kept.sum() / ho_mask.sum(), 1),
        'holdout_auc': round(roc_auc_score(y_ho, ho_probs), 3) if len(set(y_ho)) >= 2 else None,
    }
    return model, res

results = {}
for det in sorted(df['detector'].unique()):
    d = df[df['detector'] == det]
    if len(d) < MIN_TRADES_PER_DETECTOR:
        continue
    out = train_final(d, det)
    if out is None:
        print(f'{det}: skipped (insufficient data after split)'); continue
    model, res = out
    results[det] = (model, res)
    # Save
    model_path = MODELS_DIR / f'{det}_lightgbm_v1.joblib'
    meta_path = MODELS_DIR / f'{det}_lightgbm_v1_meta.json'
    joblib.dump(model, model_path)
    with open(meta_path, 'w') as f:
        json.dump({
            **res,
            'feature_names': FEATURES,
            'snapshot_date': '2026-04-27',
            'trained_at_utc': datetime.now(timezone.utc).isoformat(),
            'lgbm_params': {
                'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5,
                'num_leaves': 15, 'min_child_samples': 20,
                'reg_alpha': 0.1, 'reg_lambda': 0.1,
                'random_state': RANDOM_SEED,
            },
        }, f, indent=2, default=str)
    print(f'\n=== {det} ===')
    for k, v in res.items():
        print(f'  {k:30s}  {v}')
    print(f'  Saved: {model_path.name} + meta')

print(f'\n{len(results)} models trained and saved to {MODELS_DIR}')

## 6. Summary table — does the classifier actually lift PF?

In [ ]:
rows = []
for det, (model, res) in results.items():
    rows.append({
        'detector': det,
        'threshold': res['threshold_locked'],
        'baseline_PF': res['holdout_baseline_pf'],
        'filtered_PF': res['holdout_filtered_pf'],
        'PF_lift': round(res['holdout_filtered_pf'] - res['holdout_baseline_pf'], 2),
        'baseline_WR': res['holdout_baseline_wr'],
        'filtered_WR': res['holdout_filtered_wr'],
        'WR_lift_pts': round(res['holdout_filtered_wr'] - res['holdout_baseline_wr'], 1),
        'kept_pct': res['holdout_kept_pct'],
        'AUC': res['holdout_auc'],
        'ship_floor_1.30': '✓' if res['holdout_filtered_pf'] >= 1.30 else '✗',
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 7. Feature importance — what is the classifier actually using?

In [ ]:
for det, (model, res) in results.items():
    imp = pd.DataFrame({
        'feature': FEATURES,
        'importance': model.feature_importances_,
    }).sort_values('importance', ascending=False).head(15)
    print(f'\n=== {det} — top 15 features ===')
    print(imp.to_string(index=False))

## 8. Done — verify artifacts in Drive

After this cell completes, the following files exist on Drive at `My Drive/ai-finance/models/v8_lightgbm/`:

- `macd_pullback_long_lightgbm_v1.joblib`
- `macd_pullback_long_lightgbm_v1_meta.json`
- `macd_pullback_short_lightgbm_v1.joblib`
- `macd_pullback_short_lightgbm_v1_meta.json`
- `macd_early_trend_short_lightgbm_v1.joblib`
- `macd_early_trend_short_lightgbm_v1_meta.json`

(`rsi_recovery_long` is skipped — only 2 trades available, far below the `MIN_TRADES_PER_DETECTOR=100` threshold. Stays rule-based for now.)

**Next step:** on the VPS, pull these into the live executor via `scripts/deploy_v8_lightgbm.sh` (Phase 3 — coming next).

In [ ]:
import os
for f in sorted(os.listdir(MODELS_DIR)):
    p = MODELS_DIR / f
    print(f'{f:55s}  {p.stat().st_size / 1024:>7.1f} KB')